# 第0关：看清终点

这一关不训练模型。我们先让官方模型跑起来，认识它给出的表示，并检查它是否关注 tRNA 的天然配对。

过关条件：
- [ ] 环境可用，优先使用 MPS，失败时解释原因并退回 CPU。
- [ ] 76 nt tRNA 的序列、括号结构和 21 对碱基通过检查。
- [ ] 官方封装输出 768、76×768、76×76 三种特征。
- [ ] 用 transformers 直接加载同一权重，手工计算最后一层平均注意力，记录与封装的差异。
- [ ] 生成四张独立 PDF，并完成 1,000 个阈值的 INF 扫描。
- [ ] 保存数值、源码笔记和环境清单；整个 notebook 可从头执行。

这些复选框是任务清单，实际是否通过以末尾断言和 `scripts/check_level0.py` 为准。

## 1．先确认工具和设备

这一格读取相对路径配置，打印版本，再做一次真正的 MPS 张量运算。MPS 是让 Mac 的 GPU 帮忙计算的接口；可用性检查通过后，后面加载模型仍可能失败，因此推理格还会捕获错误并退回 CPU。只打印环境名，不公开本机用户路径。

In [ ]:
from pathlib import Path
import sys, json, platform, warnings, importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
import transformers
import yaml

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.yaml").exists(), "请在项目根目录或 notebooks 目录运行"
sys.path.insert(0, str(ROOT / "src"))
cfg = yaml.safe_load((ROOT / "config.yaml").read_text())
CHECKPOINT = ROOT / cfg["checkpoint"]
FIGURES = ROOT / cfg["figures"]
RESULTS = ROOT / cfg["results"]
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
# 官方库会给出包含加载路径的提示，下面只保留相对路径。
def public_error(exc):
    text = str(exc).replace(str(ROOT.parent), "<workspace>")
    text = text.replace(str(Path.home()), "<home>")
    return text

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
device = torch.device("cpu")
fallback_reason = None
if torch.backends.mps.is_available():
    try:
        probe = torch.ones((2, 2), device="mps") @ torch.ones((2, 2), device="mps")
        assert probe.cpu().sum().item() == 8
        device = torch.device("mps")
    except Exception as exc:
        fallback_reason = public_error(exc)
else:
    fallback_reason = "torch.backends.mps.is_available() 为 False"
print("Python:", platform.python_version())
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("设备:", device, "；CPU 回退原因:", fallback_reason or "无需回退")

## 2．准备一份能看懂的标准答案

这里使用题目给定的酵母 tRNA-Phe（PDB 1EHZ）76 nt 序列。受体臂、D 臂、反密码子臂和 T 臂构成公认的三叶草结构。由题目指定的四段配对生成括号串和接触矩阵，后面所有“模型有没有抓到配对”的检查都用这份答案。

这是把修饰核苷简写为 A/U/C/G 后的序列与指定的 21 对经典配对；它不是从 PDB 自动提取的全部三级相互作用。括号“(”与“)”表示一对，点表示该位置在这份答案中未配对。

In [ ]:
seq = "GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA"
arms = {"acceptor": (1, 7, 72), "D": (10, 13, 25),
        "anticodon": (27, 31, 43), "T": (49, 53, 65)}
pairs = [(i, j) for start, end, partner in arms.values()
         for i, j in zip(range(start, end + 1), range(partner, partner - (end-start) - 1, -1))]
assert len(seq) == 76
native = np.zeros((76, 76), dtype=np.uint8)
dots = ["."] * 76
canonical = {"AU", "UA", "GC", "CG", "GU", "UG"}
used = set()
for i, j in pairs:
    assert i < j and i not in used and j not in used
    used.update([i, j])
    assert seq[i-1] + seq[j-1] in canonical, (i, j)
    native[i-1, j-1] = native[j-1, i-1] = 1
    dots[i-1], dots[j-1] = "(", ")"
dot_bracket = "".join(dots)
stack, recovered = [], []
for k, ch in enumerate(dot_bracket, 1):
    if ch == "(": stack.append(k)
    elif ch == ")":
        assert stack, "右括号多于左括号"
        recovered.append((stack.pop(), k))
assert not stack and set(recovered) == set(pairs)
assert native.shape == (76, 76) and native.sum() == 42
print("序列:", seq)
print("结构:", dot_bracket)
print("长度:", len(seq), "碱基对数:", len(pairs))
display(pd.DataFrame([(i,j,seq[i-1]+seq[j-1]) for i,j in pairs], columns=["i_1based","j_1based","bases"]))
pd.DataFrame(native).to_csv(RESULTS / "native_contact.csv", index=False)

## 3．使用官方封装提取特征

这一格直接调用 PyPI 的官方 `structRFM_infer`。`cls_feat` 用一个 768 维向量概括整条 RNA，可用于同源或功能分类；`seq_feat` 给每个碱基一个 768 维向量，可用于位点任务；成对矩阵给每两个碱基一个分数，可用于结构任务。

**字段名需要核实：本地官方源码的 `mat_feat` 是 `seq_feat @ seq_feat.T`，而公式 7 所需的最后一层平均注意力叫 `last_mean_attn_feat`。** 我们把二者都保留下来，下一格实际验证，不把形状相同误当作含义相同。

In [ ]:
from structRFM.infer import structRFM_infer
import contextlib, io
model_paras = dict(max_length=514, dim=768, layer=12, num_attention_heads=12)
load_log = io.StringIO()
with contextlib.redirect_stdout(load_log):
    try:
        official = structRFM_infer(from_pretrained=str(CHECKPOINT), device=device, **model_paras)
        feats = official.extract_feature(seq)
    except Exception as exc:
        if device.type == "cpu":
            raise
        fallback_reason = public_error(exc)
        print("MPS 推理失败，退回 CPU:", fallback_reason)
        device = torch.device("cpu")
        official = structRFM_infer(from_pretrained=str(CHECKPOINT), device=device, **model_paras)
        feats = official.extract_feature(seq)
print(public_error(load_log.getvalue()))
print("实际使用设备:", device)
for key, value in feats.items():
    print(key, tuple(value.shape))
assert tuple(feats["cls_feat"].shape) == (768,)
assert tuple(feats["seq_feat"].shape) == (76, 768)
assert tuple(feats["mat_feat"].shape) == (76, 76)
# 先转到 CPU 保留结果，释放这一个模型，避免同时占用两份模型内存。
feat_arrays = {k: v.detach().cpu().numpy() for k,v in feats.items()}
del feats, official
import gc
gc.collect()
if device.type == "mps": torch.mps.empty_cache()

## 4．独立复算公式 7

下面用 transformers 的 BERT 掩码模型类直接加载同一目录中的权重。明确选择 eager 注意力实现，确保可以取到注意力矩阵。分词使用官方函数补上 [CLS]/[SEP]，避免两次输入不一致。

公式 7 的计算是：取第 11 层的 12 个头，去掉两个特殊 token 的行列，再求平均。不做对称化，也不把剩余行重新归一化；删除特殊 token 后，行和可能小于 1。

先和题目指定的 `mat_feat` 比较，再和封装的 `last_mean_attn_feat` 比较。前者不一致时，我们同时复算碱基特征点积，查明差异究竟来自定义还是计算错误。

In [ ]:
from transformers import BertForMaskedLM
from structRFM.data.tokenizer import get_mlm_tokenizer
from structRFM.data.RNAdata import process_mlm_input_seq

tokenizer = get_mlm_tokenizer(max_length=514)
inputs = tokenizer(process_mlm_input_seq(seq), return_tensors="pt")
assert inputs["input_ids"].shape == (1, 78)
print("特殊 token:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0, [0,-1]].tolist()))
direct = BertForMaskedLM.from_pretrained(str(CHECKPOINT), output_attentions=True,
                                        output_hidden_states=True, attn_implementation="eager")
direct.eval()
try:
    direct.to(device)
    with torch.no_grad():
        out = direct(**{k:v.to(device) for k,v in inputs.items()})
except Exception as exc:
    if device.type == "cpu": raise
    fallback_reason = public_error(exc)
    print("直接加载路径退回 CPU:", fallback_reason)
    device = torch.device("cpu")
    direct.to(device)
    with torch.no_grad(): out = direct(**inputs)
assert len(out.attentions) == 12 and out.attentions[11].shape[1] == 12
last_heads = out.attentions[11][0, :, 1:-1, 1:-1].detach().cpu().numpy()
fpair = last_heads.mean(axis=0)
head5 = last_heads[5]
mat_error = float(np.max(np.abs(fpair - feat_arrays["mat_feat"])))
mean_error = float(np.max(np.abs(fpair - feat_arrays["last_mean_attn_feat"])))
seq_direct = out.hidden_states[-1][0, 1:-1].detach().cpu().numpy()
seq_error = float(np.max(np.abs(seq_direct - feat_arrays["seq_feat"])))
gram_error = float(np.max(np.abs(feat_arrays["seq_feat"] @ feat_arrays["seq_feat"].T - feat_arrays["mat_feat"])))
print("公式 7 对 mat_feat，最大绝对误差:", mat_error)
print("公式 7 对 last_mean_attn_feat，最大绝对误差:", mean_error)
print("两次碱基特征最大误差:", seq_error)
print("碱基特征点积对 mat_feat，最大绝对误差:", gram_error)
print("去特殊 token 后行和范围:", fpair.sum(1).min(), fpair.sum(1).max())
assert mean_error < 1e-5 and seq_error < 1e-4
pd.DataFrame(fpair).to_csv(RESULTS / "fpair_mean_last_layer.csv", index=False)
pd.DataFrame(head5).to_csv(RESULTS / "attn_L11_H5.csv", index=False)
del direct, out
gc.collect()
if device.type == "mps": torch.mps.empty_cache()

### 如何解释差异

`mat_feat` 的点积分数与平均注意力本来就不是同一个量：点积可以超过 1，通常对称；注意力按方向分配，通常不对称。关键是上格 `last_mean_attn_feat` 的误差接近 0。这说明两条路径读的是相同权重、相同输入，并且正确剔除了 [CLS]/[SEP]。

点积用 CPU 的 NumPy 复算与 MPS 的矩阵乘法可能有很小的浮点误差。公式 7 的比较没有做对称化、额外归一化或更换层/头。

## 5．把模型分数与天然配对并排在脑中比较

下面生成三张独立 PDF，不使用 subplot。横纵坐标均为从 1 起计的碱基位置。请肉眼比较：注意力热图上的反对角线短条纹，是否落在天然接触图中四段茎的位置？

第 11 层第 5 头均从 0 起计。题目给出的论文 Fig. 2d 参考结果是 INF 0.462、阈值 0.540；这里先保留原始注意力，热图不做对称化。图只保存 PDF；Jupyter 中的绘图预览不另存 PNG。

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 18, 'font.family': 'Arial', 'pdf.fonttype': 42, 'ps.fonttype': 42})

def heatmap(matrix, filename, title, label):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrix, origin="lower", extent=(0.5,76.5,0.5,76.5),
                   interpolation="nearest", cmap="viridis", aspect="equal", vmin=0)
    ax.set(xlabel="Nucleotide position (1-based)", ylabel="Nucleotide position (1-based)", title=title)
    fig.colorbar(im, ax=ax, label=label, fraction=0.046, pad=0.04)
    fig.savefig(FIGURES / filename, format="pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

heatmap(fpair, "fpair_mean_last_layer.pdf", "Last-layer mean attention", "Attention")
heatmap(head5, "attn_L11_H5.pdf", "Attention: layer 11, head 5", "Attention")
heatmap(native, "native_contact_tRNA_Phe.pdf", "Native tRNA-Phe contacts", "Contact (0/1)")

## 6．最小零样本体检：先把评分流程跑通

INF = √(PPV × STY)。PPV 是“报出来的配对中有多少是真的”，STY 是“所有真实配对中找回多少”。复用模块在 `src/pairmask/metrics.py`；下面把每个阈值的中间结果留下来。

注意力有方向，天然配对没有方向。因此本次主结果先取 `max(A[i,j], A[j,i])`，表示任一方向足够强就报告这对；只统计 i<j，忽略对角线，每对只算一次，使用分数严格大于阈值。不增加 AU/GC/GU 筛选、最短环、单伙伴或非交叉解码。另存 mean 和只读上三角两种约定的扫描结果，便于看出约定的影响。

阈值为 0、0.001、…、0.999。先用几个能手算的例子检查评分，防止重复计数或空预测除零。论文参考值来自 116 条 PDB 序列；这里只有一条，而且最佳阈值在本条上选择，仅用于流程体检，不能作为泛化性能。

In [ ]:
from pairmask.metrics import contact_metrics
assert contact_metrics(native, native, 0.5)["INF"] == 1.0
assert contact_metrics(np.zeros_like(native), native, 0.5)["INF"] == 0.0
one = np.zeros_like(native); one[0,71] = 1
assert contact_metrics(one, native, 0.5)["TP"] == 1
assert contact_metrics(one, native, 0.5)["FP"] == 0
rows = []
for convention in ["max", "mean", "upper"]:
    for step in range(1000):
        threshold = step / 1000
        rows.append({"symmetrize": convention, "threshold": threshold,
                     **contact_metrics(head5, native, threshold, convention)})
scan_all = pd.DataFrame(rows)
scan_all.to_csv(RESULTS / "zeroshot_inf_scan_all_conventions.csv", index=False)
scan = scan_all.query("symmetrize == 'max'").copy()
scan.to_csv(RESULTS / "zeroshot_inf_scan_tRNA.csv", index=False)
best = scan.loc[scan["INF"].idxmax()]
print("最佳阈值（如并列，取最小值）:", best["threshold"], "最佳 INF:", best["INF"])
display(best.to_frame("value"))
print("题目参考阈值 0.540 在本条上的结果:", contact_metrics(head5, native, 0.540))
display(scan_all.loc[scan_all.groupby("symmetrize")["INF"].idxmax()])
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(scan["threshold"], scan["INF"], color="#176b87", linewidth=2)
ax.scatter([best["threshold"]], [best["INF"]], color="#ba352b", zorder=3,
           label=f"Best: t={best['threshold']:.3f}, INF={best['INF']:.3f}")
ax.set(xlabel="Attention threshold", ylabel="INF", title="Zero-shot check: one tRNA", xlim=(0,1), ylim=(0,1))
ax.legend(fontsize=13)
fig.savefig(FIGURES / "zeroshot_inf_scan_tRNA.pdf", format="pdf", bbox_inches="tight")
plt.show(); plt.close(fig)

## 7．保存数值证据并检查本关产物

这一格把设备、形状、误差和最佳分数写入 JSON，并列出四张 PDF，方便过关脚本核对。这里保存的是实际运算结果，不是事先填写的预期值。源码和数据清单在 notes 中。

In [ ]:
summary = {"sequence_length": len(seq), "native_pairs": len(pairs), "dot_bracket": dot_bracket,
           "device": str(device), "mps_available": torch.backends.mps.is_available(),
           "fallback_reason": fallback_reason,
           "versions": {x:metadata.version(x) for x in ["torch","transformers","numpy","structRFM"]},
           "feature_shapes": {k:list(v.shape) for k,v in feat_arrays.items()},
           "pair_vs_mat_max_abs_error": mat_error,
           "pair_vs_last_mean_attention_max_abs_error": mean_error,
           "seq_feature_max_abs_error": seq_error,
           "gram_vs_mat_max_abs_error": gram_error,
           "symmetrize": "max", "threshold_comparison": ">", "min_loop_filter": False,
           "best_threshold": float(best["threshold"]), "best_inf": float(best["INF"]),
           "best_counts": {k:int(best[k]) for k in ["TP","FP","FN"]}}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
expected = ["fpair_mean_last_layer.pdf", "attn_L11_H5.pdf", "native_contact_tRNA_Phe.pdf", "zeroshot_inf_scan_tRNA.pdf"]
for name in expected:
    assert (FIGURES / name).stat().st_size > 1000
    print("已生成:", "figures/level0/" + name)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 小结：这一关看到了什么

- 官方模型把一条 76 nt RNA 转成整条序列、逐碱基和成对的表示，实际形状及设备见上格。
- 公式 7 的平均注意力对应 `last_mean_attn_feat`；本次官方源码的 `mat_feat` 是另一种点积矩阵。二者不同的原因是定义，不应强行把它们改成一样。
- 三张热图让我们比较模型注意力与四段天然配对。注意力本身不是二级结构，也不保证对称、单伙伴或不交叉。
- 上格记录的最佳阈值与 INF 只适用于这条 RNA 和本项目明确写出的计分约定。它的用途是检查数据、模型和评估链路，不能直接和论文的跨序列结果画等号。
- 结构遮盖如何与普通遮盖组合、动态比例如何计步、MUSES 解码的局限等，详见 `notes/00_code_map.md`。本关没有做训练。